# TrackMania 2020 Ghost Telemetry Ingestion

Reads `.Replay.Gbx` and `.Ghost.Gbx` files from `/lakehouse/default/Files/replays/`
and creates two Delta tables:

- **silver_replay_header** — one row per file (metadata, race time, player info)
- **silver_replay_telemetry** — one row per telemetry sample (52 fields at ~50 ms intervals)

Handles both **LZO-compressed bodies** (v10 and v11 CPlugEntRecordData, used by API-downloaded `.Replay.Gbx` files) and **zlib-compressed bodies** (used by `.Ghost.Gbx` files).

> **Prerequisites:** run `fabric_replay_download.py` first, or manually upload `.Replay.Gbx` / `.Ghost.Gbx` files to `/lakehouse/default/Files/replays/`.

In [ ]:
pip install python-lzo

## Parser functions

All GBX parsing logic is self-contained here — no external package needed beyond `python-lzo`.

In [ ]:
import struct, zlib, io, math, os, re, hashlib
from datetime import datetime

try:
    import lzo
except ImportError:
    lzo = None


# ---------- low-level readers ----------

def read_uint8(f):
    return struct.unpack('<B', f.read(1))[0]

def read_uint16(f):
    return struct.unpack('<H', f.read(2))[0]

def read_int16(f):
    return struct.unpack('<h', f.read(2))[0]

def read_int32(f):
    return struct.unpack('<i', f.read(4))[0]

def read_uint32(f):
    return struct.unpack('<I', f.read(4))[0]

def read_string(f):
    length = read_uint32(f)
    if length == 0 or length > 100000:
        return ""
    data = f.read(length)
    try:
        return data.decode('utf-8')
    except UnicodeDecodeError:
        return data.decode('latin-1', errors='ignore')


# ---------- lookback string reader ----------

class _LookbackReader:
    """Manages GBX lookback string reading (string interning system)."""

    def __init__(self):
        self.id_version = None
        self.lookback_strings = {}
        self.counter = 0

    def read_id(self, f):
        if self.id_version is None:
            self.id_version = read_uint32(f)
            if self.id_version < 3:
                return ""
        index = read_uint32(f)
        if index == 0xFFFFFFFF:
            return ""
        high_bits = (index >> 30) & 0x3
        if high_bits not in (1, 2):
            return ""
        masked_index = index & 0x3FFFFFFF
        if masked_index != 0:
            return self.lookback_strings.get(masked_index, "")
        string = read_string(f)
        self.counter += 1
        self.lookback_strings[self.counter] = string
        return string

    def read_ident(self, f):
        id_str = self.read_id(f)
        collection = self.read_id(f)
        author = self.read_id(f)
        return (id_str, collection, author)


# ---------- header metadata ----------

def _parse_gbx_header_metadata(f, user_data_size):
    """Parse GBX header user-data chunks to extract replay metadata."""
    if user_data_size == 0:
        return {}

    user_data_start = f.tell()
    metadata = {}

    try:
        num_chunks = read_uint32(f)
        if num_chunks == 0 or num_chunks > 1000:
            f.seek(user_data_start + user_data_size)
            return {}

        chunk_headers = []
        for _ in range(num_chunks):
            chunk_id = read_uint32(f)
            chunk_size = read_int32(f) & 0x7FFFFFFF
            chunk_headers.append({'id': chunk_id, 'size': chunk_size})

        lookback = _LookbackReader()

        for chunk in chunk_headers:
            chunk_start = f.tell()
            try:
                if chunk['id'] == 0x03093000:
                    chunk_version = read_uint32(f)
                    if chunk_version >= 4 and chunk_version != 9999:
                        map_info = lookback.read_ident(f)
                        if map_info[0]:
                            metadata['map_uid'] = map_info[0]
                        if map_info[2]:
                            metadata['map_author'] = map_info[2]
                    time = read_int32(f)
                    if time >= 0:
                        metadata['race_time_ms'] = time
                    nickname = read_string(f)
                    if nickname:
                        metadata['player_nickname'] = nickname
                    if chunk_version >= 6:
                        login = read_string(f)
                        if login:
                            metadata['player_login'] = login
            except Exception:
                pass
            f.seek(chunk_start + chunk['size'])

    except Exception as e:
        print(f"⚠ Could not parse GBX header metadata: {e}")
    finally:
        f.seek(user_data_start + user_data_size)

    return metadata


# ---------- vehicle sample parser ----------

def parse_vehicle_vis_sample(time_ms, sample_data):
    """Parse a CSceneVehicleVis sample (107 bytes) into 52 telemetry fields."""
    if len(sample_data) != 107:
        return None

    try:
        def u8(o):  return sample_data[o]
        def i8(o):  return struct.unpack('b', sample_data[o:o+1])[0]
        def u16(o): return struct.unpack('<H', sample_data[o:o+2])[0]
        def i16(o): return struct.unpack('<h', sample_data[o:o+2])[0]
        def f32(o): return struct.unpack('<f', sample_data[o:o+4])[0]

        x, y, z = f32(47), f32(51), f32(55)

        angle = u16(59) * math.pi / 65535.0
        axis_heading = i16(61) * math.pi / 32767.0
        axis_pitch = (i16(63) / 32767.0) * (math.pi / 2.0)
        speed = math.exp(i16(65) / 1000.0)
        vel_heading = (i8(67) / 127.0) * math.pi
        vel_pitch = (i8(68) / 127.0) * (math.pi / 2.0)

        ax = math.sin(angle) * math.cos(axis_pitch) * math.cos(axis_heading)
        ay = math.sin(angle) * math.cos(axis_pitch) * math.sin(axis_heading)
        az = math.sin(angle) * math.sin(axis_pitch)
        qw = math.cos(angle)

        sinr_cosp = 2.0 * (qw * ax + ay * az)
        cosr_cosp = 1.0 - 2.0 * (ax * ax + ay * ay)
        roll = math.atan2(sinr_cosp, cosr_cosp)

        sinp = 2.0 * (qw * ay - az * ax)
        pitch = math.copysign(math.pi / 2, sinp) if abs(sinp) >= 1 else math.asin(sinp)

        yaw = math.atan2(2.0 * (qw * az + ax * ay), 1.0 - 2.0 * (ay * ay + az * az))

        vel_x = speed * math.cos(vel_pitch) * math.cos(vel_heading)
        vel_y = speed * math.cos(vel_pitch) * math.sin(vel_heading)
        vel_z = speed * math.sin(vel_pitch)

        side_speed = ((u16(2) / 65536.0) - 0.5) * 2000.0
        rpm = u8(5)

        fl_wheel_rot = (u8(6) / 255.0) * (2 * math.pi) + (u8(7) * 2 * math.pi)
        fr_wheel_rot = (u8(8) / 255.0) * (2 * math.pi) + (u8(9) * 2 * math.pi)
        rr_wheel_rot = (u8(10) / 255.0) * (2 * math.pi) + (u8(11) * 2 * math.pi)
        rl_wheel_rot = (u8(12) / 255.0) * (2 * math.pi) + (u8(13) * 2 * math.pi)

        steer = ((u8(14) / 255.0) - 0.5) * 2.0
        brake = u8(18) / 255.0
        gas = (u8(15) / 255.0) + brake
        turbo_time = u8(21) / 255.0

        fl_dampen = ((u8(23) / 255.0) - 0.5) * 4.0
        fr_dampen = ((u8(25) / 255.0) - 0.5) * 4.0
        rr_dampen = ((u8(27) / 255.0) - 0.5) * 4.0
        rl_dampen = ((u8(29) / 255.0) - 0.5) * 4.0

        fl_ground_mat, fr_ground_mat = u8(24), u8(26)
        rr_ground_mat, rl_ground_mat = u8(28), u8(30)

        is_turbo = (u8(31) & 0x82) != 0
        fl_slip = (u8(32) & 0x40) != 0
        fr_slip = (u8(33) & 0x01) != 0
        rr_slip = (u8(33) & 0x04) != 0
        rl_slip = (u8(33) & 0x10) != 0

        is_top_contact = (u8(76) & 0x20) != 0

        fl_ice = u8(81) / 255.0
        fr_ice = u8(82) / 255.0
        rr_ice = u8(83) / 255.0
        rl_ice = u8(84) / 255.0

        rf = u8(89)
        is_ground_contact = (rf & 0x01) != 0
        reactor_state = 1 if (rf & 0x04) else (2 if (rf & 0x08) else (3 if (rf & 0x10) else 0))
        reactor_boost = 1 if (rf & 0x20) else (2 if (rf & 0x40) else 0)

        rc = u8(90)
        reactor_pedal = 1 if (rc & 0x20) else (0 if (rc & 0x10) else -1)
        reactor_steer = -1 if (rc & 0x80) else (0 if (rc & 0x40) else 1)

        gear = u8(91) / 5.0
        fl_dirt = u8(93) / 255.0
        fr_dirt = u8(95) / 255.0
        rr_dirt = u8(97) / 255.0
        rl_dirt = u8(99) / 255.0
        wetness = u8(101) / 255.0
        sim_time_coef = u8(102) / 255.0

        return {
            'time_ms': time_ms, 'time_s': time_ms / 1000.0,
            'x': x, 'y': y, 'z': z,
            'speed': speed, 'side_speed': side_speed,
            'vel_x': vel_x, 'vel_y': vel_y, 'vel_z': vel_z,
            'pitch_deg': math.degrees(pitch), 'yaw_deg': math.degrees(yaw), 'roll_deg': math.degrees(roll),
            'steer': steer, 'gas': gas, 'brake': brake, 'gear': gear, 'rpm': rpm,
            'is_turbo': is_turbo, 'turbo_time': turbo_time,
            'is_ground_contact': is_ground_contact, 'is_top_contact': is_top_contact,
            'reactor_state': reactor_state, 'reactor_boost': reactor_boost,
            'reactor_pedal': reactor_pedal, 'reactor_steer': reactor_steer,
            'sim_time_coef': sim_time_coef, 'wetness': wetness,
            'fl_dampen': fl_dampen, 'fr_dampen': fr_dampen, 'rr_dampen': rr_dampen, 'rl_dampen': rl_dampen,
            'fl_ice': fl_ice, 'fr_ice': fr_ice, 'rr_ice': rr_ice, 'rl_ice': rl_ice,
            'fl_dirt': fl_dirt, 'fr_dirt': fr_dirt, 'rr_dirt': rr_dirt, 'rl_dirt': rl_dirt,
            'fl_slip': fl_slip, 'fr_slip': fr_slip, 'rr_slip': rr_slip, 'rl_slip': rl_slip,
            'fl_ground_mat': fl_ground_mat, 'fr_ground_mat': fr_ground_mat,
            'rr_ground_mat': rr_ground_mat, 'rl_ground_mat': rl_ground_mat,
            'fl_wheel_rot': fl_wheel_rot, 'fr_wheel_rot': fr_wheel_rot,
            'rr_wheel_rot': rr_wheel_rot, 'rl_wheel_rot': rl_wheel_rot,
        }
    except (struct.error, ValueError, IndexError):
        return None




# Pattern: pos{NNN}_{race_time}_{account_id}_{record_id}.Replay.Gbx
LB_PATTERN = re.compile(r'^pos(\d+)_(\d+)_([0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12})_(.+)\.Replay\.Gbx$')

def extract_path_metadata(filepath):
    """Extract map_uid from folder and position/race_time/account_id from filename."""
    parts = filepath.replace("\\", "/").split("/")
    file_name = parts[-1]
    meta = {}

    # map_uid is the folder name under leaderboard/
    if "leaderboard" in parts:
        idx = parts.index("leaderboard")
        if idx + 1 < len(parts) - 1:
            meta['map_uid'] = parts[idx + 1]
            meta['source'] = 'leaderboard'

    m = LB_PATTERN.match(file_name)
    if m:
        meta['position'] = int(m.group(1))
        meta['race_time_ms'] = int(m.group(2))
        meta['account_id'] = m.group(3)

    return meta


# ---------- GBX replay parser ----------

def parse_gbx_replay(filepath):
    """Parse .Replay.Gbx or .Ghost.Gbx → metadata + vehicle telemetry samples."""
    with open(filepath, 'rb') as f:
        data = f.read()

    if data[:3] != b'GBX':
        return None

    # Header layout: magic(3) + version(2) + format(1) + ref_comp(1) + body_comp(1)
    #                + unknown(1) + classID(4) = 13 bytes, then user_data_size(4)
    uds = struct.unpack_from('<I', data, 13)[0]
    after_ud = 17 + uds

    # Extract header metadata (map uid, player info, race time)
    ud_reader = io.BytesIO(data[17:17 + uds])
    header_metadata = _parse_gbx_header_metadata(ud_reader, uds)

    # Reference table: num_external(u32); if > 0, ancestor_level(u32) follows
    num_ext = struct.unpack_from('<I', data, after_ud)[0]
    body_off = (after_ud + 8) if num_ext > 0 else (after_ud + 4)

    # Body: uncompressed_size(u32) + compressed_size(u32) + data
    bu = struct.unpack_from('<I', data, body_off)[0]
    bc = struct.unpack_from('<I', data, body_off + 4)[0]
    body_raw = data[body_off + 8: body_off + 8 + bc]

    # Decompress body — LZO for .Replay.Gbx, zlib for .Ghost.Gbx
    try:
        if lzo is None:
            raise ImportError("python-lzo not installed")
        body = lzo.decompress(body_raw, False, bu)
    except Exception:
        try:
            body = zlib.decompress(body_raw)
        except Exception:
            return None

    # Find CPlugEntRecordData (0x0911F000) — doubled marker preferred
    fp = body.find(b'\x00\xf0\x11\x09\x00\xf0\x11\x09')
    if fp >= 0:
        off = fp + 8
    else:
        fp = body.find(b'\x00\xf0\x11\x09')
        if fp < 0:
            return None
        off = fp + 4

    fb = io.BytesIO(body[off:])
    ver = struct.unpack('<I', fb.read(4))[0]
    u_inner = struct.unpack('<I', fb.read(4))[0]
    c_inner = struct.unpack('<I', fb.read(4))[0]
    try:
        record = zlib.decompress(fb.read(c_inner))
    except Exception:
        return None

    fr = io.BytesIO(record)
    start_time = struct.unpack('<i', fr.read(4))[0]
    end_time = struct.unpack('<i', fr.read(4))[0]

    # Entity descriptors
    ec = struct.unpack('<I', fr.read(4))[0]
    descs = []
    for _ in range(ec):
        cls = struct.unpack('<I', fr.read(4))[0]
        ss = struct.unpack('<i', fr.read(4))[0]
        fr.read(4); fr.read(4)
        dl = struct.unpack('<I', fr.read(4))[0]
        fr.read(dl); fr.read(4)
        descs.append((cls, ss))

    # Skip notices
    nc = struct.unpack('<I', fr.read(4))[0]
    for _ in range(nc):
        fr.read(12)

    # Parse entities
    samples = []

    while fr.tell() < len(record) - 1:
        he = struct.unpack('<B', fr.read(1))[0]
        if he != 1:
            break

        if ver >= 11:
            # Version 11: columnar delta-encoded storage
            eidx = struct.unpack('<i', fr.read(4))[0]
            fr.read(4)   # u01
            fr.read(4)   # e_start
            fr.read(4)   # e_end
            fr.read(4)   # u04
            ns = struct.unpack('<I', fr.read(4))[0]
            nk = struct.unpack('<I', fr.read(4))[0]
            fr.read(4)   # u07

            cls = descs[eidx][0] if 0 <= eidx < len(descs) else 0
            is_vehicle = (cls == 0x0A018000)

            # Time deltas
            times = [0]
            for _ in range(ns - 1):
                times.append(times[-1] + struct.unpack('<I', fr.read(4))[0])

            # Delta-decoded columns
            columns = []
            for k in range(nk):
                col = list(fr.read(ns))
                for i in range(1, len(col)):
                    col[i] = (col[i - 1] + struct.unpack('b', bytes([col[i]]))[0]) & 0xFF
                columns.append(col)

            if is_vehicle:
                for s in range(ns):
                    raw = bytes(columns[k][s] for k in range(nk))
                    parsed = parse_vehicle_vis_sample(times[s], raw[:107])
                    if parsed:
                        samples.append(parsed)

            # hasNext + samples2 loop
            hn = struct.unpack('<B', fr.read(1))[0]
            if hn == 1:
                while True:
                    hs2 = struct.unpack('<B', fr.read(1))[0]
                    if hs2 != 1:
                        break
                    fr.read(4); fr.read(4)
                    dl = struct.unpack('<I', fr.read(4))[0]
                    fr.read(dl)

        else:
            # Version 10: entity_index + has_sample loops
            eidx = struct.unpack('<i', fr.read(4))[0]
            fr.read(16)  # u01–u04

            cls = descs[eidx][0] if 0 <= eidx < len(descs) else 0
            is_vehicle = (cls == 0x0A018000)

            while True:
                hs = struct.unpack('<B', fr.read(1))[0]
                if hs != 1:
                    break
                t_ms = struct.unpack('<i', fr.read(4))[0]
                sl = struct.unpack('<I', fr.read(4))[0]
                sd = fr.read(sl)
                if is_vehicle:
                    parsed = parse_vehicle_vis_sample(t_ms, sd)
                    if parsed:
                        samples.append(parsed)

            # hasNext + samples2
            fr.read(1)
            while True:
                hs2 = struct.unpack('<B', fr.read(1))[0]
                if hs2 != 1:
                    break
                fr.read(8)
                dl = struct.unpack('<I', fr.read(4))[0]
                fr.read(dl)

    return {
        'start_time': start_time,
        'end_time': end_time,
        'num_samples': len(samples),
        'samples': samples,
        'record_version': ver,
        'header_metadata': header_metadata,
    }

## Read and parse files

Find all `.Replay.Gbx` and `.Ghost.Gbx` files in the lakehouse, parse each one, and collect results.

In [ ]:
input_dir = "/lakehouse/default/Files/replays/"

gbx_files = []
for root, dirs, files in os.walk(input_dir):
    for f in files:
        if f.endswith(".Replay.Gbx") or f.endswith(".Ghost.Gbx"):
            gbx_files.append(os.path.join(root, f))

print(f"Found {len(gbx_files)} .Replay.Gbx / .Ghost.Gbx files")

parsed_replays = []

for filepath in gbx_files:
    try:
        result = parse_gbx_replay(filepath)
        if result:
            file_name = os.path.basename(filepath)
            replay_id = hashlib.md5(file_name.encode()).hexdigest()
            parsed_replays.append({
                'replay_id': replay_id,
                'file_name': file_name,
                'start_time': result['start_time'],
                'end_time': result['end_time'],
                'num_samples': result['num_samples'],
                'samples': result['samples'],
                'record_version': result['record_version'],
                'header_metadata': result['header_metadata'],
                'path_metadata': extract_path_metadata(filepath),
            })
            print(f"✓ {file_name}: ver={result['record_version']}, {result['num_samples']} samples")
        else:
            print(f"✗ Failed to parse {os.path.basename(filepath)}")
    except Exception as e:
        print(f"✗ Error parsing {os.path.basename(filepath)}: {e}")

print(f"\nSuccessfully parsed {len(parsed_replays)} files")

## Create silver_replay_header table

One row per file with race metadata.

In [ ]:
if parsed_replays:
    header_rows = []
    for r in parsed_replays:
        meta = r['header_metadata']
        pmeta = r.get('path_metadata', {})

        # Path metadata fills in blanks for leaderboard replays
        map_uid = meta.get('map_uid') or pmeta.get('map_uid', '')
        source = pmeta.get('source', 'player')
        race_time_ms = meta.get('race_time_ms') or pmeta.get('race_time_ms', r['end_time'] - r['start_time'])
        if race_time_ms < 0:
            print(f"⚠ Negative race_time_ms ({race_time_ms}) for {r['file_name']} — clamping to 0")
        race_time_ms = max(int(race_time_ms), 0)

        header_rows.append({
            'replay_id': str(r['replay_id']),
            'file_name': str(r['file_name']),
            'source': source,
            'map_uid': str(map_uid),
            'map_author': str(meta.get('map_author', '')),
            'player_nickname': str(meta.get('player_nickname', '')),
            'player_login': str(meta.get('player_login', '')),
            'account_id': str(pmeta.get('account_id', '')),
            'position': int(pmeta.get('position', 0)),
            'race_time_ms': race_time_ms,
            'race_time_s': round(race_time_ms / 1000.0, 3),
            'start_time_ms': int(r['start_time']),
            'end_time_ms': int(r['end_time']),
            'num_samples': int(r['num_samples']),
            'sample_period_ms': int(50),
            'ingested_at': datetime.now(),
        })

    df_header = spark.createDataFrame(header_rows)
    df_header.write.format("delta").mode("overwrite").saveAsTable("silver_replay_header")
    print(f"✓ Wrote {len(header_rows)} rows to silver_replay_header")
    df_header.show(5, truncate=False)
else:
    print("No files to ingest")

## Create silver_replay_telemetry table

One row per telemetry sample (52 fields). Written in batches of 100 000 rows.

In [ ]:
if parsed_replays:
    telemetry_rows = []

    for r in parsed_replays:
        rid = str(r['replay_id'])
        for s in r['samples']:
            telemetry_rows.append({
                'replay_id': rid,
                'time_ms': int(s['time_ms']),
                'time_s': float(s['time_s']),
                'x': float(s['x']), 'y': float(s['y']), 'z': float(s['z']),
                'speed': float(s['speed']),
                'side_speed': float(s['side_speed']),
                'vel_x': float(s['vel_x']), 'vel_y': float(s['vel_y']), 'vel_z': float(s['vel_z']),
                'pitch_deg': float(s['pitch_deg']),
                'yaw_deg': float(s['yaw_deg']),
                'roll_deg': float(s['roll_deg']),
                'steer': float(s['steer']),
                'gas': float(s['gas']),
                'brake': float(s['brake']),
                'gear': float(s['gear']),
                'rpm': int(s['rpm']),
                'is_turbo': bool(s['is_turbo']),
                'turbo_time': float(s['turbo_time']),
                'is_ground_contact': bool(s['is_ground_contact']),
                'is_top_contact': bool(s['is_top_contact']),
                'reactor_state': int(s['reactor_state']),
                'reactor_boost': int(s['reactor_boost']),
                'reactor_pedal': int(s['reactor_pedal']),
                'reactor_steer': int(s['reactor_steer']),
                'sim_time_coef': float(s['sim_time_coef']),
                'wetness': float(s['wetness']),
                'fl_dampen': float(s['fl_dampen']), 'fr_dampen': float(s['fr_dampen']),
                'rr_dampen': float(s['rr_dampen']), 'rl_dampen': float(s['rl_dampen']),
                'fl_ice': float(s['fl_ice']), 'fr_ice': float(s['fr_ice']),
                'rr_ice': float(s['rr_ice']), 'rl_ice': float(s['rl_ice']),
                'fl_dirt': float(s['fl_dirt']), 'fr_dirt': float(s['fr_dirt']),
                'rr_dirt': float(s['rr_dirt']), 'rl_dirt': float(s['rl_dirt']),
                'fl_slip': bool(s['fl_slip']), 'fr_slip': bool(s['fr_slip']),
                'rr_slip': bool(s['rr_slip']), 'rl_slip': bool(s['rl_slip']),
                'fl_ground_mat': int(s['fl_ground_mat']), 'fr_ground_mat': int(s['fr_ground_mat']),
                'rr_ground_mat': int(s['rr_ground_mat']), 'rl_ground_mat': int(s['rl_ground_mat']),
                'fl_wheel_rot': float(s['fl_wheel_rot']), 'fr_wheel_rot': float(s['fr_wheel_rot']),
                'rr_wheel_rot': float(s['rr_wheel_rot']), 'rl_wheel_rot': float(s['rl_wheel_rot']),
            })

    print(f"Prepared {len(telemetry_rows)} telemetry rows")

    batch_size = 100000
    for i in range(0, len(telemetry_rows), batch_size):
        batch = telemetry_rows[i:i + batch_size]
        df_t = spark.createDataFrame(batch)
        write_mode = "overwrite" if i == 0 else "append"
        df_t.write.format("delta").mode(write_mode).saveAsTable("silver_replay_telemetry")
        print(f"✓ Batch {i // batch_size + 1}: {len(batch)} rows ({write_mode})")

    print(f"\n✓ Total telemetry rows: {len(telemetry_rows)}")
    spark.table("silver_replay_telemetry").show(10, truncate=False)
else:
    print("No telemetry data to ingest")

## Summary

In [ ]:
print("=" * 60)
print("INGESTION SUMMARY")
print("=" * 60)
print(f"Files found:            {len(gbx_files)}")
print(f"Files parsed:           {len(parsed_replays)}")
print(f"Total samples ingested: {sum(r['num_samples'] for r in parsed_replays)}")
print("=" * 60)

print("\nReplay Header Table:")
spark.table("silver_replay_header").show(10, truncate=False)

print("\nReplay Telemetry Table (sample):")
spark.sql("""
    SELECT replay_id, time_s, x, y, z, speed, steer, gas, brake
    FROM silver_replay_telemetry
    LIMIT 20
""").show(20, truncate=False)